# House Prices — Compact Top-10 Feature Model

This notebook builds a **compact version** of the tuned stacked ensemble from the main
pipeline notebook (`house-prices-end-to-end-regression-pipeline.ipynb`), using **only the
10 most important features**.

Workflow:
1. Load the saved production model (`../models/final_stack_model.joblib`) and rank features by importance.
2. Rebuild those 10 features from the raw CSVs with the exact same preprocessing
   (reusing the transform constants stored inside the artifact).
3. Train the same stack architecture (same tuned hyperparameters from `../models/best_params.txt`)
   on a held-out split for honest metrics.
4. Retrain on all labeled rows and save a self-contained artifact: **`../models/top10_stack_model.joblib`**.

Requires (produced by the main notebook): `../data/train.csv`, `../data/test.csv`,
`../models/best_params.txt`, `../models/final_stack_model.joblib`.


In [1]:
# Standard library
import json
import os
import sys

# Third-party
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (
    GradientBoostingRegressor,
    BaggingRegressor,
    VotingRegressor,
    StackingRegressor,
)
from sklearn.linear_model import RidgeCV

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

SEED = 42


## 1. Rank features with the loaded production model

Importance is read from the fitted base models inside `../models/final_stack_model.joblib`:

- **CatBoost branch** — its built-in feature importances.
- **Voting branch** — XGB and Bagging(GB) importances mixed with the blend weights stored
  inside the fitted voting model (`[w_xgb, 1 - w_xgb]`). Each bagged GB was trained on a
  random subset of columns, so its importances are mapped back to the full feature list first.

Every source is normalized to sum to 1 before averaging, so the scales are comparable.


In [2]:
art = joblib.load("../models/final_stack_model.joblib")
full_model = art["model"]
feat_names = pd.Index(art["feature_columns"])

def normalized_importance(values):
    s = pd.Series(values, index=feat_names)
    return s / s.sum()

imp_catboost = normalized_importance(
    full_model.named_estimators_["catboost"].feature_importances_
)

voting_fitted = full_model.named_estimators_["voting_xgb_bgb"]
imp_xgb = normalized_importance(
    voting_fitted.named_estimators_["xgb"].feature_importances_
)

bagging_fitted = voting_fitted.named_estimators_["bagging_gb"]
bag_totals = np.zeros(len(feat_names))
for gb_est, gb_cols in zip(bagging_fitted.estimators_, bagging_fitted.estimators_features_):
    bag_totals[gb_cols] += gb_est.feature_importances_
imp_bagging = normalized_importance(bag_totals)

w_xgb = voting_fitted.weights[0]
stack_importance = (imp_catboost + (w_xgb * imp_xgb + (1 - w_xgb) * imp_bagging)) / 2

top10_features = stack_importance.nlargest(10).index.tolist()

print("Top 10 features by loaded-model importance:")
for rank, feat in enumerate(top10_features, 1):
    print(f"  {rank:2d}. {feat:<22s} {stack_importance[feat]:6.1%}")


Top 10 features by loaded-model importance:
   1. OverallQual             25.6%
   2. GrLivArea               10.7%
   3. ExterQual                8.9%
   4. GarageCars               7.3%
   5. KitchenQual              5.1%
   6. TotalBsmtSF              4.8%
   7. YearBuilt                2.9%
   8. 1stFlrSF                 2.9%
   9. GarageFinish             2.8%
  10. FullBath                 2.6%


## 2. Rebuild the 10 features from the raw CSVs

The artifact stores every preprocessing constant of the main pipeline (categorical NA fills,
ordinal mappings, one-hot layout, training medians), so the features built here are
guaranteed to match the main notebook exactly. `build_features` mirrors the artifact's
`predict_from_raw` transform, then keeps only the wanted columns.


In [3]:
def build_features(raw_df, artifact, columns):
    """Raw CSV rows -> model-ready feature matrix (same transform as the main pipeline)."""
    df = raw_df.drop(columns=artifact["drop_columns"]).fillna(artifact["cat_na_fills"])
    for col, mapping in artifact["ordinal_mappings"].items():
        df[col] = df[col].map(mapping)
    df = pd.get_dummies(df, columns=artifact["nominal_columns"], dtype=int)
    df = df.fillna(artifact["train_medians"])
    return df.reindex(columns=columns, fill_value=0)

train_raw = pd.read_csv("../data/train.csv")
y = train_raw["SalePrice"]
X_top10 = build_features(train_raw, art, top10_features)

print(f"Feature matrix: {X_top10.shape[0]} rows x {X_top10.shape[1]} columns")
X_top10.head()


Feature matrix: 1460 rows x 10 columns


,OverallQual,GrLivArea,ExterQual,GarageCars,KitchenQual,TotalBsmtSF,YearBuilt,1stFlrSF,GarageFinish,FullBath
0,7,1710,4,2,4,856,2003,856,2,2
1,6,1262,3,2,3,1262,1976,1262,2,2
2,7,1786,4,2,4,920,2001,920,2,2
3,7,1717,3,3,4,756,1915,961,1,1
4,8,2198,4,3,4,1145,2000,1145,2,2


## 3. Train the compact stack on a held-out split

Same 90/10 split (`random_state=42`), same stack architecture and tuned hyperparameters as
the main notebook — only the input columns change. For reference, the full 205-feature stack
scores about **RMSE 28,253 / R² 0.913** on this same split (see the main notebook).

One caveat: the importance ranking comes from a model that saw all labeled rows, so the test
score carries a small feature-selection bias; the model weights themselves never see the
test rows.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top10, y, test_size=0.1, random_state=SEED
)

# Scale the columns with more than 2 unique values (same rule as the main notebook)
scale_top10 = [col for col in top10_features if X_train[col].nunique() > 2]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_top10] = scaler.fit_transform(X_train[scale_top10])
X_test_scaled[scale_top10] = scaler.transform(X_test[scale_top10])

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Scaled columns: {scale_top10}")


Train: (1314, 10) | Test: (146, 10)
Scaled columns: ['OverallQual', 'GrLivArea', 'ExterQual', 'GarageCars', 'KitchenQual', 'TotalBsmtSF', 'YearBuilt', '1stFlrSF', 'GarageFinish', 'FullBath']


In [5]:
# Tuned hyperparameters frozen by the main notebook's Optuna studies
with open("../models/best_params.txt") as f:
    _saved = json.load(f)

CATBOOST_PARAMS = _saved["catboost"]
VOTING_PARAMS = _saved["voting"]
print(f"Loaded best params from ../models/best_params.txt (w_xgb = {VOTING_PARAMS['w_xgb']:.3f})")


def build_voting_model(p):

    xgb_model = XGBRegressor(
        n_estimators=p["xgb_n_estimators"],
        learning_rate=p["xgb_learning_rate"],
        max_depth=p["xgb_max_depth"],
        min_child_weight=p["xgb_min_child_weight"],
        gamma=p.get("xgb_gamma", 0.0),
        subsample=p["xgb_subsample"],
        colsample_bytree=p["xgb_colsample_bytree"],
        colsample_bylevel=p.get("xgb_colsample_bylevel", 1.0),
        colsample_bynode=p.get("xgb_colsample_bynode", 1.0),
        reg_alpha=p["xgb_reg_alpha"],
        reg_lambda=p["xgb_reg_lambda"],
        max_delta_step=p.get("xgb_max_delta_step", 0),
        max_bin=p.get("xgb_max_bin", 256),
        grow_policy=p.get("xgb_grow_policy", "depthwise"),
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED
    )

    bagging_gb_model = BaggingRegressor(
        estimator=GradientBoostingRegressor(
            n_estimators=p["gb_n_estimators"],
            learning_rate=p["gb_learning_rate"],
            loss=p.get("gb_loss", "squared_error"),
            max_depth=p["gb_max_depth"],
            min_samples_split=p.get("gb_min_samples_split", 2),
            min_samples_leaf=p["gb_min_samples_leaf"],
            max_features=p.get("gb_max_features", None),
            subsample=p["gb_subsample"],
            validation_fraction=p.get("gb_validation_fraction", 0.1),
            n_iter_no_change=p.get("gb_n_iter_no_change", None),
            tol=p.get("gb_tol", 1e-4),
            random_state=SEED
        ),
        n_estimators=p["bag_n_estimators"],
        max_samples=p["bag_max_samples"],
        max_features=p["bag_max_features"],
        bootstrap=p.get("bag_bootstrap", True),
        bootstrap_features=p.get("bag_bootstrap_features", False),
        random_state=SEED,
        n_jobs=-1
    )

    return VotingRegressor(
        estimators=[
            ("xgb", xgb_model),
            ("bagging_gb", bagging_gb_model)
        ],
        weights=[p["w_xgb"], 1 - p["w_xgb"]],
        n_jobs=1
    )


def make_top10_stack():
    """Same stack architecture as the main notebook's final model."""
    return StackingRegressor(
        estimators=[
            (
                "catboost",
                CatBoostRegressor(
                    **CATBOOST_PARAMS,
                    loss_function="RMSE",
                    verbose=0,
                    random_seed=SEED
                )
            ),
            (
                "voting_xgb_bgb",
                build_voting_model(VOTING_PARAMS)
            )
        ],
        final_estimator=RidgeCV(alphas=np.logspace(-4, 3, 30)),
        cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
        n_jobs=1
    )


Loaded best params from ../models/best_params.txt (w_xgb = 0.000)


In [6]:
stack_top10 = make_top10_stack()
stack_top10.fit(X_train_scaled, y_train)

test_pred = stack_top10.predict(X_test_scaled)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print(f"Top-10 stack | Test RMSE: {rmse(y_test, test_pred):.4f} | "
      f"MAE: {mean_absolute_error(y_test, test_pred):.4f} | "
      f"R²: {r2_score(y_test, test_pred):.4f}")

print("\nStack weights (Ridge coefficients):")
for name, coef in zip(stack_top10.named_estimators_, stack_top10.final_estimator_.coef_):
    print(f"  {name}: {coef:.4f}")


Top-10 stack | Test RMSE: 32066.3560 | MAE: 16967.4319 | R²: 0.8875

Stack weights (Ridge coefficients):
  catboost: 0.4289
  voting_xgb_bgb: 0.6156


## 4. Retrain on all labeled rows and save the artifact

Like the main pipeline, the shipped model is retrained on **every** labeled row. The saved
artifact bundles the model, its own 10-column scaler, and all preprocessing constants, so
`../models/top10_stack_model.joblib` can predict straight from raw `test.csv`-style rows with no
other files.


In [7]:
scaler_full = StandardScaler()
X_full_scaled = X_top10.copy()
X_full_scaled[scale_top10] = scaler_full.fit_transform(X_top10[scale_top10])

final_top10_model = make_top10_stack()
final_top10_model.fit(X_full_scaled, y)
print(f"Final top-10 stack trained on all {len(X_full_scaled)} labeled rows")

import sklearn
import xgboost
import catboost

top10_artifact = {
    "model": final_top10_model,
    "scaler": scaler_full,
    "scale_features": scale_top10,
    "feature_columns": top10_features,
    "feature_importance": stack_importance.nlargest(10),  # provenance: why these 10
    "train_medians": art["train_medians"],
    "nominal_columns": art["nominal_columns"],
    "ordinal_mappings": art["ordinal_mappings"],
    "cat_na_fills": art["cat_na_fills"],
    "drop_columns": art["drop_columns"],
    "versions": {
        "python": sys.version.split()[0],
        "scikit-learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "catboost": catboost.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

joblib.dump(top10_artifact, "../models/top10_stack_model.joblib", compress=3)
print(f"Saved ../models/top10_stack_model.joblib "
      f"({os.path.getsize('../models/top10_stack_model.joblib') / 1e6:.1f} MB)")


Final top-10 stack trained on all 1460 labeled rows
Saved ../models/top10_stack_model.joblib (1.3 MB)


In [8]:
# --- Reload the saved artifact and verify the round trip on raw test.csv ---
loaded = joblib.load("../models/top10_stack_model.joblib")
print("Artifact saved with:", loaded["versions"])

def predict_top10_from_raw(raw_df, artifact):
    df = build_features(raw_df, artifact, artifact["feature_columns"])
    df[artifact["scale_features"]] = artifact["scaler"].transform(df[artifact["scale_features"]])
    return artifact["model"].predict(df)

raw_test = pd.read_csv("../data/test.csv")
preds_in_session = predict_top10_from_raw(raw_test, top10_artifact)
preds_reloaded = predict_top10_from_raw(raw_test, loaded)

print(f"Reloaded predictions match in-session model: "
      f"{np.allclose(preds_reloaded, preds_in_session)}")
print(f"Max abs difference: {np.max(np.abs(preds_reloaded - preds_in_session)):.6f}")
print("\nFirst 5 predictions from raw test.csv:")
print(np.round(preds_reloaded[:5], 2))


Artifact saved with: {'python': '3.13.9', 'scikit-learn': '1.8.0', 'xgboost': '3.3.0', 'catboost': '1.2.10', 'pandas': '3.0.1', 'numpy': '2.4.0'}


Reloaded predictions match in-session model: True
Max abs difference: 0.000000

First 5 predictions from raw test.csv:
[118314.59 158259.49 174141.49 188078.14 209144.32]
